# Model Fitting


In [1]:
from types import SimpleNamespace
import os
from IPython import get_ipython
import lightning as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from torchinfo import summary

from meta import append_meta

from data_module import NPZTensorDataModule
from model import MultiLayerLeakyReLUModel
from callbacks import LossRatioMonitor

detect if runnig as notebook or script


In [2]:
if get_ipython():
    is_running_in_notebook = True
    print("executing as notebook ...")
else:
    is_running_in_notebook = False
    print("executing as script ...")

executing as notebook ...


In [3]:
args = SimpleNamespace(
    # io
    data_path="data/demo",
    logs_path="data/logs",
    experiment_name="demo_experiment",
    version="v1",
    # data
    batch_size=64,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
    # model
    hidden_layer_sizes=[128, 64],
    leaky_relu_factor=0.1,
    # optimizer
    learning_rate=0.01,
    weight_decay=0.0,
    # trainer
    max_steps=1_000,
    log_every_n_steps=1,
    accelerator="auto",
    devices=None,
    profiler=None,
    # callbacks
    early_stopping_val_train_ratio_training_window_size=1000,
    early_stopping_val_train_ratio_validation_window_size=100,
    early_stopping_val_train_ratio_upper_threshold=1.2,
    early_stopping_val_train_ratio_bad_epochs_limit=100,
)

In [4]:
version_path = os.path.join(args.logs_path, args.experiment_name, args.version)
append_meta(version_path, vars(args))

## load data


In [5]:
data = NPZTensorDataModule(
    path=args.data_path,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    persistent_workers=args.persistent_workers,
    pin_memory=args.pin_memory,
)

## initialize model


In [6]:
model = MultiLayerLeakyReLUModel(
    input_shape=data.input_shape[-1],
    output_shape=data.output_shape[-1],
    hidden_layer_sizes=args.hidden_layer_sizes,
    leaky_relu_factor=args.leaky_relu_factor,
    learning_rate=args.learning_rate,
    weight_decay=args.weight_decay,
)

model_summary = summary(model, input_size=(args.batch_size, data.input_shape[-1]))
display(model_summary)

append_meta(
    version_path,
    {
        "total_params": model_summary.total_params,
        "trainable_params": model_summary.trainable_params,
        "total_param_bytes": model_summary.total_param_bytes,
        "finished_fitting": False,
    },
)

Layer (type:depth-idx)                   Output Shape              Param #
MultiLayerLeakyReLUModel                 [64, 3]                   --
├─Sequential: 1-1                        [64, 3]                   --
│    └─Linear: 2-1                       [64, 128]                 384
│    └─LeakyReLU: 2-2                    [64, 128]                 --
│    └─Linear: 2-3                       [64, 64]                  8,256
│    └─LeakyReLU: 2-4                    [64, 64]                  --
│    └─Linear: 2-5                       [64, 3]                   195
Total params: 8,835
Trainable params: 8,835
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.57
Input size (MB): 0.00
Forward/backward pass size (MB): 0.10
Params size (MB): 0.04
Estimated Total Size (MB): 0.14

## Trainer


In [7]:
# loggers
logger = TensorBoardLogger(
    save_dir=args.logs_path, name=args.experiment_name, version=args.version
)

# callbacks
callbacks = [
    ModelCheckpoint(
        monitor="val/loss",
        mode="min",
        save_top_k=1,
        dirpath=logger.log_dir,
        filename="best",
    ),
    # EarlyStopping(monitor="val/loss", patience=20, mode="min"),
    LossRatioMonitor(
        training_window_size=args.early_stopping_val_train_ratio_training_window_size,
        validation_window_size=args.early_stopping_val_train_ratio_validation_window_size,
        ratio_upper_threshold=args.early_stopping_val_train_ratio_upper_threshold,
        bad_epochs_limit=args.early_stopping_val_train_ratio_bad_epochs_limit,
    ),
]

# trainer
trainer = pl.Trainer(
    max_steps=args.max_steps,
    logger=logger,
    log_every_n_steps=args.log_every_n_steps,
    callbacks=callbacks,
    accelerator=args.accelerator,
    **(dict(devices=args.devices) if args.devices else {}),
    enable_progress_bar=is_running_in_notebook,
    profiler=args.profiler,
)

/data/dust/user/kwasniok/torch-lightning-template/venv/lib/python3.13/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data/dust/user/kwasniok/torch-lightning-template/ve ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


## Fitting


In [8]:
trainer.fit(model, data)

2025-10-27 21:55:38.357085: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/data/dust/user/kwasniok/torch-lightning-template/venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /data/dust/user/kwasniok/torch-lightning-template/data/logs/demo_experiment/v1 exists and is not empty.

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | loss_fn  | MSELoss    | 0      | train
1 | sequence | Sequential | 8.8 K  | train
------------------------------------------------
8.8 K     Trainable params
0         Non-trainable params
8.8 K     Total params
0.035     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/data/dust/user/kwasniok/torch-lightning-template/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:665: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.


In [9]:
append_meta(
    version_path,
    {
        "finished_fitting": True,
    },
)
print("done")

done
